# Limpieza, análisis y estructuración de datos de Instacart con Pandas

**Actividad:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas (Instacart)
**Aprendiz:** Liliana Sofia Panesso Mendez

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este notebook desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*: ejemplos, laboratorio guiado y reto de cada una. Cada lección parte de las tablas que deja la anterior, por eso el notebook debe ejecutarse completo y de arriba hacia abajo (*Kernel → Restart Kernel and Run All Cells*).

> Las rutas del material (`datos/crudos`, `datos/procesados`) se adaptaron a la estructura del proyecto en inglés: `data/raw` y `data/processed`.

## Preparación

In [1]:
# P.1 Librerías y rutas
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

print(pd.__version__)

3.1.0


Se importaron las librerías y se definieron las rutas de entrada (`data/raw`) y salida (`data/processed`). La versión de pandas es **3.1.0**.

In [2]:
# P.2 Carga de las cinco tablas
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

Se cargaron las cinco tablas con `sep=';'`, porque los archivos usan punto y coma como separador.

In [3]:
# P.3 Verificación de tamaños (Lección 1)
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Los tamaños **coinciden** con los esperados en la Lección 1: `orders (478967, 6)`, `order_prods (4545007, 4)`, `products (49694, 4)`, `aisles (134, 2)` y `departments (21, 2)`.

## Lección 2: Limpieza y estandarización de datos crudos

Se resuelven los problemas encontrados en la exploración (actividad EV04 y Lección 1): duplicados, tipos de dato, nulos, texto sin estandarizar y duplicados implícitos. Para cada problema se decide entre **eliminar, corregir o conservar**, siguiendo este orden: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

### Ejemplos de la lección

#### Duplicados exactos

In [4]:
# L2.E1 Ver los duplicados antes de borrarlos
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Se muestran **30** filas: 15 pedidos y sus 15 copias, como se esperaba. Todas tienen `order_dow = 3` y `order_hour_of_day = 2`: que ocurran en la misma franja apunta a un fallo del sistema y no a pedidos reales, así que es seguro borrarlas.

In [5]:
# L2.E2 Eliminar los duplicados
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


Tras `drop_duplicates()` el resultado es **`(478952, 6) True`**, igual al esperado: quedan 478 952 pedidos y `order_id` ya es único.

#### Tipos de dato con `astype`

In [6]:
# L2.E3 Corregir tipos y reducir memoria
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


La memoria de `orders` bajó de **23,0 MB** a **7,2 MB**, como indica el material. Se usó `Int16` (con mayúscula) en las columnas con nulos, porque admite enteros y nulos a la vez.

#### Nulos: comprobar la causa antes de decidir

In [7]:
# L2.E4 ¿Los nulos de days_since_prior_order son primeros pedidos?
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


El resultado es **`True`**: cada nulo de `days_since_prior_order` es un primer pedido. Decisión: **conservar** el nulo; rellenarlo con 0 inventaría que el cliente volvió el mismo día.

In [8]:
# L2.E5 Productos sin nombre
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Hay una sola combinación, pasillo **100** y departamento **21** (`missing`), con **1 258** productos; después de `fillna` quedan **0** nulos. Coincide con lo esperado. Decisión: **corregir** con `'Unknown'`, porque el producto existe y se usa en pedidos.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 (ejercicio 3) | Conservar el nulo |

#### Texto con los métodos `.str`

In [9]:
# L2.E6 Estandarizar los nombres de producto
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


Había **403** nombres con espacios dobles y **16** con espacio invisible (`\xa0`); después de limpiar quedan **0 0**. Coincide con el material.

#### Duplicados implícitos

In [10]:
# L2.E7 Nombres repetidos después de estandarizar
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


Aparecen **141** nombres repetidos tras estandarizar y **268 de 280** productos implicados aparecen en pedidos, igual que en el material. Decisión: **conservar y documentar**, porque tienen `product_id` distintos; en los análisis se agrupa por `product_id`.

#### Limpieza reutilizable con funciones

In [11]:
# L2.E8 Funciones de limpieza
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


La función devuelve **`(478952, 6)`**: aplicada a una tabla ya limpia no cambia nada, como se esperaba.

In [12]:
# L2.E9 Verificar toda la limpieza
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Verificación superada')

Verificación superada


Los cuatro `assert` pasaron sin error (se agregó un `print` para dejar constancia visible). Hay **28 817** nulos en `days_since_prior_order` porque 2 de los nulos originales estaban entre los duplicados.

### Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Se construye un **pipeline**: una secuencia de funciones encadenadas que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso. Se trabaja sobre copias nuevas, sin tocar las tablas ya limpias.

#### Paso 1: recargar los datos crudos

In [13]:
# L2.LAB1 Recargar los datos crudos
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


Se recargaron los archivos crudos: **`(478967, 6) (49694, 4) (4545007, 4)`**, los tamaños originales.

#### Paso 2: una función que anota el estado de la tabla

In [14]:
# L2.LAB2 Función de registro
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

`anotar` guarda una "foto" (filas, nulos, memoria) y devuelve la tabla **sin cambios**, para poder intercalarla entre pasos.

#### Paso 3: un paso de limpieza por función

In [15]:
# L2.LAB3 Pasos de limpieza
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

Cada función hace un solo paso de limpieza y recibe y devuelve un DataFrame; esa regla permite encadenarlas.

#### Paso 4: encadenar los pasos con `pipe`

In [16]:
# L2.LAB4 Pipeline
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

El pipeline se lee de arriba abajo como una receta: crudo → quitar duplicados → anotar → corregir tipos → anotar. La celda se ejecutó una sola vez, por eso `registro` no tiene filas repetidas.

#### Paso 5: el informe de cambios

In [17]:
# L2.LAB5 Informe de cambios
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


El informe **coincide** con la tabla esperada del material:

- Quitar duplicados eliminó **15 filas** y **2 nulos** en `orders` (478 967 → 478 952; 28 819 → 28 817): dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de **23,0 a 7,2 MB** y la de `order_prods` de **145,4 a 54,5 MB**.
- Los **1 258** nulos de `products` pasaron a 0 al rellenar con `'Unknown'`.
- Los **836** nulos de `order_prods` se conservan a propósito.

#### Paso 6: validar el resultado

In [18]:
# L2.LAB6 Validación
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


Resultado: **`Validación superada`** y **`True True`**, como se esperaba: el pipeline produce exactamente las mismas tablas que la limpieza manual de los ejemplos.

### Reto 2

#### Ejercicio 1
Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si `reordered` solo contiene 0 y 1.

In [19]:
# L2.R1 Duplicados y valores de reordered en order_prods
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?:', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1?: True


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (**0**) ni pares `order_id` + `product_id` repetidos (**0**), y `reordered` solo contiene **0 y 1**. No necesita limpieza de duplicados.

#### Ejercicio 2
Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [20]:
# L2.R2 Nombres de pasillos y departamentos
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nombres duplicados:', texto.duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())
    print('  con espacio invisible:', texto.str.contains('\xa0', regex=False).sum())

aisles
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con espacio invisible: 0
departments
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con espacio invisible: 0


**Respuesta:** ni `aisles` ni `departments` tienen nombres duplicados, espacios al inicio o al final, espacios dobles ni espacios invisibles (**0** en todos los casos). No necesitan limpieza de texto.

#### Ejercicio 3
Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [21]:
# L2.R3 Pedidos con nulos en add_to_cart_order
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
print('Nulos:', nulos_carrito.sum())
print('Pedidos con nulos:', len(pedidos_con_nulos))

tamano = order_prods[order_prods['order_id'].isin(pedidos_con_nulos)].groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', tamano.min())
print('Máximo de add_to_cart_order registrado en toda la tabla:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos en toda la tabla:', (order_prods.groupby('order_id').size() > 64).sum())

Nulos: 836
Pedidos con nulos: 70
Mínimo de productos en esos pedidos: 65
Máximo de add_to_cart_order registrado en toda la tabla: 64


Pedidos con más de 64 productos en toda la tabla: 70


**Respuesta:** los **836** nulos están en **70** pedidos, y cada uno tiene como mínimo **65** productos. El valor máximo registrado de `add_to_cart_order` es **64** y hay exactamente **70** pedidos con más de 64 productos: el sistema solo registró la posición hasta la 64, y los productos posteriores quedaron sin posición. Por eso se conservan como nulos.

#### Ejercicio 4
Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [22]:
# L2.R4 Nombres que cambiaron con la estandarización
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
tenia_nombre = products_crudo['product_name'].notna()
cambiaron = products_crudo.loc[tenia_nombre, 'product_name'] != products.loc[tenia_nombre, 'product_name']
print('Nombres no nulos:', tenia_nombre.sum())
print('Nombres que cambiaron:', cambiaron.sum())
print(f'Porcentaje: {cambiaron.mean() * 100:.2f} %')

Nombres no nulos: 48436
Nombres que cambiaron: 6321
Porcentaje: 13.05 %


**Respuesta:** de los **48 436** nombres no nulos, **6 321** (**13,05 %**) cambiaron con la estandarización (espacios y mayúsculas iniciales).

#### Ejercicio 5
Crea una Series con el nombre del día de cada pedido y conviértela a una categoría **ordenada** de domingo a sábado. Compara la memoria del texto con la de la categoría.

In [23]:
# L2.R5 Día de la semana como categoría ordenada
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
orden_dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

dia_texto = orders['order_dow'].map(dias)
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=orden_dias, ordered=True))

print('Tipo texto:', dia_texto.dtype, '| memoria:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Tipo categoría:', dia_categoria.dtype, '| memoria:', round(dia_categoria.memory_usage(deep=True) / 1e6, 2), 'MB')
print('¿Ordenada?:', dia_categoria.cat.ordered)
print('Primera y última categoría:', dia_categoria.min(), '-', dia_categoria.max())

Tipo texto: str | memoria: 7.14 MB
Tipo categoría: category | memoria: 0.48 MB
¿Ordenada?: True
Primera y última categoría: domingo - sábado


**Respuesta:** la Series de texto ocupa **7,14 MB** y la categoría ordenada **0,48 MB**, unas 15 veces menos, porque guarda cada nombre de día una sola vez. Al estar ordenada, el mínimo es *domingo* y el máximo *sábado*.

#### Ejercicio 6
¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [24]:
# L2.R6 Productos 'Unknown' en los pedidos
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Productos Unknown en el catálogo:', len(ids_unknown))
print('Filas de order_prods con productos Unknown:', len(filas_unknown))
print(f'Porcentaje de filas: {len(filas_unknown) / len(order_prods) * 100:.2f} %')
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())

Productos Unknown en el catálogo: 1258
Filas de order_prods con productos Unknown: 9547
Porcentaje de filas: 0.21 %
Pedidos en los que aparecen: 8254


**Respuesta:** hay **9 547** filas de `order_prods` (**0,21 %**) con productos `'Unknown'`, repartidas en **8 254** pedidos. **No los eliminaría**: son compras reales; borrarlas quitaría productos de 8 254 pedidos y alteraría conteos como el tamaño de los pedidos o la tasa de recompra. Basta con identificarlos.

#### Ejercicio 7
En una **copia** de `departments`, reemplaza `'missing'` por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [25]:
# L2.R7 'missing' como nulo real en una copia
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print('Nulos en la copia:', departments_na['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())
departments_na.tail(3)

Nulos en la copia: 1
Nulos en la original: 0


,department_id,department
18,19,snacks
19,20,deli
20,21,NaN


En la copia, `'missing'` pasó a ser un nulo real (**1** nulo), mientras que la tabla original sigue con **0** nulos.

In [26]:
# L2.R7b Efecto de un nulo al agrupar
prueba = products.merge(departments_na, on='department_id', how='left')
print('Productos por departamento (original con missing):',
      products.merge(departments, on='department_id').groupby('department').size().sum())
print('Productos por departamento (copia con NA):', prueba.groupby('department').size().sum())
print('Productos que desaparecen del groupby:', prueba['department'].isna().sum())

Productos por departamento (original con missing): 49694
Productos por departamento (copia con NA): 48436
Productos que desaparecen del groupby: 1258


**Respuesta:** no conviene hacerlo en la tabla original porque `groupby` descarta los nulos por defecto: en esta prueba, **1 258** productos desaparecen del agrupamiento (49 694 frente a 48 436), y los totales dejarían de cuadrar con el número real de productos.

### Conclusiones de la Lección 2

- **Duplicados:** se eliminaron 15 pedidos repetidos, todos del día 3 a las 2 h. `orders` quedó con 478 952 filas y `order_id` único.
- **Tipos:** con `astype` la memoria bajó de 23,0 a 7,2 MB en `orders` y de 145,4 a 54,5 MB en `order_prods`. Los tipos `Int16` permiten conservar los nulos.
- **Nulos:** cada nulo se decidió según su causa: se conservaron los 28 817 de primeros pedidos y los 836 de posiciones mayores a 64 en el carrito, y se rellenaron con `'Unknown'` los 1 258 nombres faltantes.
- **Texto:** la estandarización cambió 6 321 nombres y dejó 141 duplicados implícitos, que se conservan porque tienen `product_id` distintos.
- **Pipeline:** el Laboratorio 2 reproduce la limpieza manual con funciones encadenadas (`True True`) y deja un informe de cambios con cifras.

## Lección 3: Filtrado, ordenamiento y agrupación de datos

Las tablas limpias se convierten en respuestas: cuándo compran los clientes y con qué frecuencia vuelven. **Filtrar** elige las filas que importan, **ordenar** las jerarquiza y **agrupar** las resume. Cada pregunta se traduce primero a la forma "*métrica* por *grupo*".

### Ejemplos de la lección

#### Filtrado con máscaras booleanas

In [27]:
# L3.E1 Una condición
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


La máscara devuelve valores `True`/`False` y el filtro deja **28 817** primeros pedidos, como se esperaba (ya sin los 2 primeros pedidos que estaban duplicados).

In [28]:
# L3.E2 Condiciones combinadas
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


Hay **45 599** pedidos de fin de semana (días 0 y 6) hechos entre las 6 y las 11 h. Coincide con el material.

#### Filtrado con `query`

In [29]:
# L3.E3 query
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


`query` devuelve **1 449** pedidos hechos desde las 10 p. m. por clientes que tardaron 30 días en volver. Coincide con el material.

#### Ordenamiento con `sort_values` y `nlargest`

In [30]:
# L3.E4 Ordenamiento
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


Los tres clientes con más pedidos tienen `order_number` **100**, y en `historial` el cliente **2** aparece primero con sus pedidos 1 y 4, como indica el material.

#### Agrupación con `groupby`

In [31]:
# L3.E5 Pedidos por hora
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


Resultado **`10 40578`**: la hora pico es las 10 a. m., igual que en el material.

In [32]:
# L3.E6 Agregación con nombre por día
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


La tabla **coincide** con la del material (por ejemplo, el día 0 tiene 84 090 pedidos, 57 661 clientes y una espera media de 11,8 días). El día 3 tiene 60 897 pedidos (no 60 912 como en la EV04) porque allí estaban los 15 duplicados eliminados. Según la convención del material, el día 0 es domingo.

#### Tabla dinámica con `pivot_table`

In [33]:
# L3.E7 Matriz hora x día
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


La matriz tiene forma **(24, 7)**: 24 horas por 7 días.

In [34]:
# L3.E7b Celda más alta de la matriz
hora_max, dia_max = matriz.stack().idxmax()
print('Hora:', int(hora_max), '| día:', int(dia_max), '| pedidos:', matriz.stack().max())

Hora: 10 | día: 1 | pedidos: 7881


La celda más alta es la hora **10** del día **1** (lunes), con **7 881** pedidos, como indica el material.

#### Valores por fila con `transform`

In [35]:
# L3.E8 transform
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


Resultado **`231946 30527`**: 30 527 clientes con 5 pedidos o más, que suman 231 946 pedidos. Coincide con el material. La columna auxiliar se eliminó al final de la celda.

### Laboratorio guiado 3: perfil de hábitos de compra

Mercadeo quiere saber **a qué hora compra cada tipo de cliente** según qué tan seguido compra. Plan: (1) contar los pedidos de cada cliente, (2) clasificar a los clientes en tres segmentos, (3) clasificar los pedidos en franjas horarias y (4) cruzar segmentos y franjas.

#### Paso 1: pedidos por cliente

In [36]:
# L3.LAB1 Pedidos por cliente
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

Hay **157 437** clientes, con una mediana de **2** pedidos y un máximo de **28**, como se esperaba.

#### Paso 2: segmentar clientes con `pd.cut`

In [37]:
# L3.LAB2 Segmentos de clientes
import numpy as np

segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64


order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


Los segmentos **coinciden** con el material: ocasional **55 357** (35,2 %), regular **71 553** (45,4 %) y frecuente **30 527** (19,4 %).

#### Paso 3: llevar el segmento a cada pedido con `map`

In [38]:
# L3.LAB3 Segmento en cada pedido
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


Cada pedido tiene ahora su segmento (`ocasional`, `regular` o `frecuente`), buscado por `user_id` con `map`.

#### Paso 4: resumen por segmento

In [39]:
# L3.LAB4 Resumen por segmento
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


La tabla **coincide** con la del material. El **19,4 %** de los clientes (los frecuentes) hace casi la mitad de los pedidos (**231 946** de 478 952) y vuelve cada **6** días (mediana), frente a 11 de los regulares y 16 de los ocasionales.

#### Paso 5: franjas horarias y tabla cruzada

In [40]:
# L3.LAB5 Franjas y perfil
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


El perfil **coincide** con el material. Todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (**35,2 %** frente a **30,0 %** de los ocasionales). Para mercadeo, una campaña de fidelización debería llegar por la mañana.

#### Paso 6: las combinaciones de día y franja más activas

In [41]:
# L3.LAB6 Día y franja más activos
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

Las combinaciones más activas son domingo en la tarde (**42 340**), lunes en la tarde (**37 432**) y martes en la tarde (**30 751**), como en el material.

#### Paso 7: dejar `orders` como estaba

In [42]:
# L3.LAB7 Quitar columnas auxiliares
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


Se eliminaron las columnas auxiliares y `orders` vuelve a **(478952, 6)**, lista para la Lección 4.

### Reto 3

#### Ejercicio 1
¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

**Métrica por grupo:** número de pedidos **por hora**, con la condición `order_dow == 0` (domingo).

In [43]:
# L3.R1 Horas con más pedidos el domingo
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64

**Respuesta:** los domingos, las horas con más pedidos son las **15 h** (7 585), las **14 h** (7 533) y las **13 h** (7 483). El domingo el pico se desplaza a la tarde, mientras que en el total de la semana es a las 10 h.

#### Ejercicio 2
¿A qué hora es mayor la espera media entre pedidos (`days_since_prior_order`) y a qué hora es menor?

**Métrica por grupo:** media de `days_since_prior_order` **por hora**.

In [44]:
# L3.R2 Espera media por hora
espera_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print('Hora con mayor espera media:', espera_hora.idxmax(), '->', round(espera_hora.max(), 2), 'días')
print('Hora con menor espera media:', espera_hora.idxmin(), '->', round(espera_hora.min(), 2), 'días')

Hora con mayor espera media: 0 -> 12.52 días
Hora con menor espera media: 22 -> 10.62 días


**Respuesta:** la espera media es mayor a las **0 h** (**12,52 días**) y menor a las **22 h** (**10,62 días**).

#### Ejercicio 3
¿Cuántos clientes tienen 10 pedidos o más?

**Métrica por grupo y condición:** número de pedidos **por cliente**, y luego contar los clientes con 10 o más.

In [45]:
# L3.R3 Clientes con 10 pedidos o más
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
print('Clientes con 10 pedidos o más:', (pedidos_cliente >= 10).sum())
print(f'Porcentaje de clientes: {(pedidos_cliente >= 10).mean() * 100:.2f} %')

Clientes con 10 pedidos o más: 6327
Porcentaje de clientes: 4.02 %


**Respuesta:** **6 327** clientes (**4,02 %**) tienen 10 pedidos o más en estos datos.

#### Ejercicio 4
¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

**Condición:** `days_since_prior_order == 7`.

In [46]:
# L3.R4 Pedidos a los 7 días
a_los_7 = orders[orders['days_since_prior_order'] == 7]
print('Pedidos a los 7 días:', len(a_los_7))
print(f'Porcentaje sobre el total de pedidos ({len(orders)}): {len(a_los_7) / len(orders) * 100:.2f} %')
print(f"Porcentaje sobre los pedidos con pedido anterior ({orders['days_since_prior_order'].count()}): "
      f"{len(a_los_7) / orders['days_since_prior_order'].count() * 100:.2f} %")

Pedidos a los 7 días: 44577
Porcentaje sobre el total de pedidos (478952): 9.31 %
Porcentaje sobre los pedidos con pedido anterior (450135): 9.90 %


**Respuesta:** **44 577** pedidos se hicieron exactamente 7 días después del anterior: el **9,31 %** del total de 478 952 pedidos, o el **9,90 %** de los 450 135 que tienen pedido anterior.

#### Ejercicio 5
Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

**Métrica por grupo y condición:** número de productos **por pedido**, y luego contar los pedidos con 20 o más.

In [47]:
# L3.R5 Tamaño de los pedidos
productos_por_pedido = order_prods.groupby('order_id')['product_id'].count()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Respuesta:** **49 420** pedidos tienen 20 productos o más. El pedido más grande es el **61355**, con **127** productos.

#### Ejercicio 6
Usando `order_prods`: calcula la tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón ves?

**Métrica por grupo:** media de `reordered` **por posición en el carrito**, con la condición `add_to_cart_order` entre 1 y 5.

In [48]:
# L3.R6 Tasa de recompra por posición en el carrito
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta:** la tasa de recompra baja a medida que avanza la posición en el carrito: **0,679** en la posición 1, 0,677, 0,659, 0,638 y **0,617** en la posición 5. Los clientes agregan primero los productos que ya conocen.

#### Ejercicio 7
Compara la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.

**Métrica por grupo:** mediana de `days_since_prior_order` **por tipo de día** (fin de semana / entre semana).

In [49]:
# L3.R7 Mediana de espera: fin de semana frente a entre semana
fin_de_semana = orders['order_dow'].isin([0, 6]).rename('fin_de_semana')
orders.groupby(fin_de_semana)['days_since_prior_order'].agg(['median', 'mean', 'count'])

,median,mean,count
fin_de_semana,,,
False,7.0,10.874753,312767
True,8.0,11.618798,137368


**Respuesta:** la mediana de espera es **8 días** en los pedidos de fin de semana y **7 días** entre semana (medias de 11,62 y 10,87). Quienes compran el fin de semana tardan un poco más en volver.

### Conclusiones de la Lección 3

- **Horario:** la hora pico es las 10 a. m. (40 578 pedidos), y la combinación más fuerte de día y hora es el lunes a las 10 (7 881). Por franjas, todos los segmentos compran sobre todo en la tarde, y el domingo en la tarde es el momento más activo (42 340).
- **Clientes:** el 19,4 % de los clientes (frecuentes, con 5 pedidos o más) genera 231 946 de los 478 952 pedidos y vuelve con una mediana de 6 días. Los frecuentes compran más por la mañana (35,2 %).
- **Frecuencia de compra:** 44 577 pedidos (9,31 %) llegan exactamente a los 7 días, y la espera es algo mayor en fin de semana (mediana de 8 frente a 7 días).
- **Carrito:** los primeros productos del carrito son los más recomprados (0,679 en la posición 1 frente a 0,617 en la 5).
- **Técnicas:** las máscaras, `query`, `groupby` + `agg`, `pivot_table`, `transform`, `pd.cut`, `map` y `crosstab` permitieron responder preguntas de negocio en pocos pasos.

## Lección 4: Combinación y estructuración de múltiples datasets

Se unen las cinco tablas limpias en una **tabla maestra**, en la que cada fila es un producto dentro de un pedido con su nombre, pasillo, departamento y los datos del pedido. `merge` une **columnas** emparejando claves y `concat` apila **filas**. Como un `merge` mal hecho no da error (multiplica o pierde filas en silencio), cada unión se comprueba.

### Ejemplos de la lección

#### Tipos de unión con el parámetro `how`

In [50]:
# L4.E1 inner, left y outer
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



Resultado **`inner 3`**, **`left 4`** y **`outer 5`**, como se esperaba. El pedido 1 aparece dos veces porque tiene dos productos; con `left` aparece el pedido 3 sin producto y con `outer` también el pedido 4, que no tiene cliente.

#### Validación de la relación con `validate`

In [51]:
# L4.E2 validate detecta claves repetidas
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


`validate='m:1'` detectó la clave repetida: **`Detectado: Merge keys are not unique in right dataset; not a many-to-one merge`**, el mismo mensaje del material.

#### Auditoría de la unión con `indicator`

In [52]:
# L4.E3 indicator
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

Resultado **`both 450046`**, **`left_only 28906`** y **`right_only 0`**, igual que en el material. Hay 28 906 pedidos (6 %) sin productos registrados: con un `inner` habrían desaparecido sin aviso.

#### Construcción de la tabla maestra

In [53]:
# L4.E4 Tabla maestra
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio

+ products    (4545007, 7)


+ aisles      (4545007, 8)


+ departments (4545007, 9)


+ orders      (4545007, 14)


La tabla maestra mantiene siempre **4 545 007** filas y pasa de **7** a **8**, **9** y finalmente **14** columnas, como se esperaba. El `assert` confirma que no se crearon ni perdieron filas.

#### Preguntas sobre la tabla maestra

In [54]:
# L4.E5 Productos más pedidos y resumen por departamento
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
display(top)
por_depto.head(3)

,product_id,product_name,veces
0,24852,Banana,66050
1,13176,Bag Of Organic Bananas,53297
2,21137,Organic Strawberries,37039
3,21903,Organic Baby Spinach,33971
4,47209,Organic Hass Avocado,29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


**Banana** es el producto más pedido (**66 050**), seguido de **Bag Of Organic Bananas** (**53 297**). En `por_depto`: `produce` con **1 328 302** artículos y tasa **0,651**, `dairy eggs` con **759 842** y **0,671**, y `snacks` con **404 850** y **0,574**. Todo coincide con el material.

#### Apilado con `concat`

In [55]:
# L4.E6 concat
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


La tabla apilada tiene forma **(155, 3)**: 134 pasillos y 21 departamentos, como se esperaba.

#### Formato ancho y largo con `pivot_table` y `melt`

In [56]:
# L4.E7 Ancho y largo
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


Resultado **`(24, 7) (168, 3)`**: las mismas 168 cifras en formato ancho y largo, como indica el material.

#### Guardado del resultado

In [57]:
# L4.E8 Guardar la tabla maestra y el resumen por departamento
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)
maestra.to_parquet(SALIDA / 'instacart_maestra.parquet', index=False)
por_depto.to_csv(SALIDA / 'resumen_departamentos.csv')
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'instacart_maestra.parquet', 'resumen_departamentos.csv']


Se guardaron `instacart_maestra.parquet` y `resumen_departamentos.csv` en `data/processed/`.

### Laboratorio guiado 4: tabla de análisis por producto

Compras quiere una tabla con **una fila por producto**: cuántas veces se pidió, cuántos clientes lo compran, qué tan seguido se recompra y en qué posición suele ir en el carrito, para saber cuál es el producto más fiel de cada departamento. Patrón: **agregar** la maestra a otro nivel, **volver a unir** las descripciones y **clasificar dentro de cada grupo**.

#### Paso 1: agregar la tabla maestra por producto

In [58]:
# L4.LAB1 Agregar por producto
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


La tabla por producto tiene forma **(45573, 5)**: una fila por cada producto pedido alguna vez. El producto 1 se pidió **280** veces por **206** clientes, con una tasa de recompra de **0,564**. Coincide con el material.

#### Paso 2: recuperar nombre y departamento con `merge`

In [59]:
# L4.LAB2 Ficha de productos
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

La ficha tiene forma **(45573, 7)** y el producto más pedido es **Banana**: `produce`, **66 050** veces, **33 274** clientes, tasa de recompra **0,844** y posición media **4,87**. Coincide con el material.

#### Paso 3: filtrar los productos con suficientes datos

In [60]:
# L4.LAB3 Productos con al menos 1 000 pedidos
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


Quedan **739** productos con al menos 1 000 pedidos, como se esperaba. Así se evita calcular tasas de recompra con muy pocos datos.

#### Paso 4: clasificar dentro de cada departamento con `rank`

In [61]:
# L4.LAB4 Líder de cada departamento con rank
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


Hay **18** líderes (departamentos con al menos un producto de 1 000 pedidos) y los cinco primeros coinciden con el material: Whole Organic Omega 3 Milk (**0,863**), Banana (**0,844**), Extra Fancy Unsalted Mixed Nuts (**0,822**), Zero Calorie Cola (**0,818**) y Organic Honey Sweet Whole Wheat Bread (**0,799**). El producto más pedido no siempre es el más fiel: la leche Omega 3 tiene solo 1 286 pedidos, pero quien la compra casi siempre la repite.

#### Paso 5: la misma respuesta con `sort_values` y `head`

In [62]:
# L4.LAB5 Líder de cada departamento con sort_values + head
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))

18 True


Resultado **`18 True`**: la segunda forma (`sort_values` + `groupby().head(1)`) obtiene exactamente los mismos líderes.

#### Paso 6: guardar la ficha de productos

In [63]:
# L4.LAB6 Guardar la ficha y los líderes
ficha_productos.to_parquet(SALIDA / 'ficha_productos.parquet', index=False)
lideres.to_csv(SALIDA / 'lideres_por_departamento.csv', index=False)
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'ficha_productos.parquet', 'instacart_maestra.parquet', 'lideres_por_departamento.csv', 'resumen_departamentos.csv']


Se guardaron `ficha_productos.parquet` y `lideres_por_departamento.csv`. En `data/processed/` quedan los cuatro archivos generados.

### Reto 4

Todos los ejercicios usan la tabla `maestra`.

#### Ejercicio 1
Entre los pasillos con al menos 10 000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?

In [64]:
# L4.R1 Tasa de recompra por pasillo
por_pasillo = (maestra.groupby('aisle')
                      .agg(articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
grandes = por_pasillo[por_pasillo['articulos'] >= 10000]
print('Pasillos con al menos 10 000 artículos:', len(grandes), 'de', len(por_pasillo))
print()
print('Tres con mayor tasa de recompra:')
print(grandes.nlargest(3, 'tasa_recompra').round(3))
print()
print('Tasa más baja entre esos pasillos:')
print(grandes.nsmallest(1, 'tasa_recompra').round(3))
print()
print('Tasa más baja entre todos los pasillos:')
print(por_pasillo.nsmallest(1, 'tasa_recompra').round(3))

Pasillos con al menos 10 000 artículos: 77 de 134

Tres con mayor tasa de recompra:
                               articulos  tasa_recompra
aisle                                                  
milk                              125522          0.781
water seltzer sparkling water     117133          0.731
fresh fruits                      510101          0.718

Tasa más baja entre esos pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152

Tasa más baja entre todos los pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152


**Respuesta:** de los 134 pasillos, **77** tienen al menos 10 000 artículos. Entre ellos, las tasas de recompra más altas son **milk** (**0,781**), **water seltzer sparkling water** (**0,731**) y **fresh fruits** (**0,718**). La más baja es **spices seasonings** (**0,152**), que también es la más baja de todos los pasillos: las especias se compran una vez y duran mucho.

#### Ejercicio 2
Compara la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?

In [65]:
# L4.R2 Participación de departamentos: madrugada frente a total
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]
participacion = pd.DataFrame({
    'madrugada_pct': madrugada['department'].value_counts(normalize=True) * 100,
    'total_pct': maestra['department'].value_counts(normalize=True) * 100,
}).round(2)
participacion['diferencia'] = participacion['madrugada_pct'] - participacion['total_pct']

top3_madrugada = set(participacion['madrugada_pct'].nlargest(3).index)
top3_total = set(participacion['total_pct'].nlargest(3).index)
print('Artículos de madrugada:', len(madrugada))
print('Top 3 madrugada:', participacion['madrugada_pct'].nlargest(3).index.tolist())
print('Top 3 total:', participacion['total_pct'].nlargest(3).index.tolist())
print('En el top 3 de madrugada pero no en el total:', top3_madrugada - top3_total)
participacion.sort_values('madrugada_pct', ascending=False).head(6)

Artículos de madrugada: 83310
Top 3 madrugada: ['produce', 'dairy eggs', 'beverages']
Top 3 total: ['produce', 'dairy eggs', 'snacks']
En el top 3 de madrugada pero no en el total: {'beverages'}


,madrugada_pct,total_pct,diferencia
department,,,
produce,30.21,29.23,0.98
dairy eggs,15.87,16.72,-0.85
beverages,8.49,8.26,0.23
snacks,7.89,8.91,-1.02
frozen,7.65,6.91,0.74
pantry,6.17,5.78,0.39


**Respuesta:** en la madrugada (**83 310** artículos) el top 3 es produce (30,21 %), dairy eggs (15,87 %) y **beverages** (8,49 %), mientras que en el total es produce, dairy eggs y snacks. **Beverages** entra en el top 3 de madrugada pero no en el total, y snacks baja (7,89 % frente a 8,91 %).

#### Ejercicio 3
¿Cuáles son los tres productos más pedidos del departamento `snacks`?

In [66]:
# L4.R3 Productos más pedidos de snacks
(maestra[maestra['department'] == 'snacks']
    .groupby(['product_id', 'product_name']).size()
    .nlargest(3).reset_index(name='veces'))

,product_id,product_name,veces
0,13870,Lightly Salted Baked Snap Pea Crisps,3462
1,34448,Original Veggie Straws,2331
2,31040,Sea Salt Pita Chips,2275


**Respuesta:** los tres productos más pedidos de `snacks` son **Lightly Salted Baked Snap Pea Crisps** (3 462), **Original Veggie Straws** (2 331) y **Sea Salt Pita Chips** (2 275).

#### Ejercicio 4
Construye una tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?

In [67]:
# L4.R4 Tabla por cliente
por_cliente = (maestra.groupby('user_id')
                      .agg(pedidos=('order_id', 'nunique'),
                           articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
print('Clientes:', len(por_cliente))
print('Promedio de artículos por cliente:', round(por_cliente['articulos'].mean(), 2))
print('Cliente con más artículos:', por_cliente['articulos'].idxmax(),
      'con', por_cliente['articulos'].max(), 'artículos')
por_cliente.loc[[por_cliente['articulos'].idxmax()]]

Clientes: 149626
Promedio de artículos por cliente: 30.38
Cliente con más artículos: 103634 con 580 artículos


,pedidos,articulos,tasa_recompra
user_id,,,
103634,18,580,0.822414


**Respuesta:** la tabla tiene **149 626** clientes (menos que los 157 437 de `orders`, porque algunos solo tienen pedidos sin detalle de productos). El promedio es **30,38** artículos por cliente, y el cliente **103634** es el que más compró: **580** artículos en 18 pedidos, con una tasa de recompra de 0,822.

#### Ejercicio 5
Une `orders` con los pedidos de `order_prods` usando `how='inner'`. ¿Cuántas filas obtienes y por qué son menos que con `how='left'`?

In [68]:
# L4.R5 inner frente a left
pedidos_con_detalle = order_prods[['order_id']].drop_duplicates()
union_inner = orders.merge(pedidos_con_detalle, on='order_id', how='inner', validate='1:1')
union_left = orders.merge(pedidos_con_detalle, on='order_id', how='left', validate='1:1')
print('inner:', len(union_inner))
print('left :', len(union_left))
print('Diferencia:', len(union_left) - len(union_inner))

inner: 450046
left : 478952
Diferencia: 28906


**Respuesta:** con `how='inner'` se obtienen **450 046** filas y con `how='left'` **478 952**. La diferencia de **28 906** son los pedidos de `orders` sin productos en `order_prods`: `inner` solo conserva las claves presentes en ambas tablas, mientras que `left` conserva todos los pedidos.

#### Ejercicio 6
Con `concat`, arma una sola tabla con los tres productos más pedidos de `produce` y los tres de `dairy eggs`, con una columna que indique el departamento.

In [69]:
# L4.R6 concat de los top 3 de dos departamentos
def top3(depto):
    return (maestra[maestra['department'] == depto]
            .groupby(['product_id', 'product_name']).size()
            .nlargest(3).reset_index(name='veces')
            .assign(department=depto))

top_produce_lacteos = pd.concat([top3('produce'), top3('dairy eggs')], ignore_index=True)
top_produce_lacteos

,product_id,product_name,veces,department
0,24852,Banana,66050,produce
1,13176,Bag Of Organic Bananas,53297,produce
2,21137,Organic Strawberries,37039,produce
3,27845,Organic Whole Milk,19600,dairy eggs
4,49235,Organic Half & Half,10752,dairy eggs
5,27086,Half & Half,9588,dairy eggs


**Respuesta:** la tabla apilada tiene 6 filas: en `produce`, Banana (66 050), Bag Of Organic Bananas (53 297) y Organic Strawberries (37 039); en `dairy eggs`, Organic Whole Milk (19 600), Organic Half & Half (10 752) y Half & Half (9 588). La columna `department` indica el origen de cada fila.

#### Ejercicio 7
Crea una tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de `alcohol`?

In [70]:
# L4.R7 Tabla dinámica departamento x día
articulos_depto_dia = maestra.pivot_table(index='department', columns='order_dow',
                                          values='product_id', aggfunc='count')
print('Forma:', articulos_depto_dia.shape)
print('Día con más artículos de alcohol:', int(articulos_depto_dia.loc['alcohol'].idxmax()),
      'con', articulos_depto_dia.loc['alcohol'].max(), 'artículos')
articulos_depto_dia.loc[['alcohol']]

Forma: (21, 7)
Día con más artículos de alcohol: 5 con 3814 artículos


order_dow,0,1,2,3,4,5,6
department,,,,,,,
alcohol,2397,3025,2868,3094,3656,3814,2787


**Respuesta:** la tabla dinámica tiene forma **(21, 7)**: 21 departamentos por 7 días. Los artículos de `alcohol` se piden más el día **5** (**3 814** artículos; viernes según la convención del material), seguido del día 4 (3 656).

### Archivos generados en `data/processed/`

In [71]:
# L4.F Archivos procesados: filas, columnas y tamaño
archivos = {
    'instacart_maestra.parquet': maestra,
    'resumen_departamentos.csv': por_depto,
    'ficha_productos.parquet': ficha_productos,
    'lideres_por_departamento.csv': lideres,
}
pd.DataFrame([
    {'archivo': nombre, 'filas': df.shape[0], 'columnas': df.shape[1],
     'tamano_mb': round((SALIDA / nombre).stat().st_size / 1e6, 2)}
    for nombre, df in archivos.items()
])

,archivo,filas,columnas,tamano_mb
0,instacart_maestra.parquet,4545007,14,103.62
1,resumen_departamentos.csv,21,2,0.00
2,ficha_productos.parquet,45573,7,1.69
3,lideres_por_departamento.csv,18,8,0.00


Tamaños de los archivos generados:

- **`instacart_maestra.parquet`:** 4 545 007 filas, 14 columnas y **103,62 MB**. Como supera los 50 MB **no se versiona** (está en `.gitignore`). Se regenera ejecutando este notebook.
- **`resumen_departamentos.csv`:** 21 filas y 2 columnas, más `department` como índice.
- **`ficha_productos.parquet`:** 45 573 filas, 7 columnas y 1,69 MB.
- **`lideres_por_departamento.csv`:** 18 filas y 8 columnas.

Los tres archivos menores de 50 MB se versionan en Git.

### Conclusiones de la Lección 4

- **Combinación validada:** la tabla maestra une las cinco tablas en 4 545 007 filas y 14 columnas. Con `how='left'` y `validate='m:1'` en cada paso no se crearon ni perdieron filas, y `indicator=True` reveló 28 906 pedidos sin productos que un `inner` habría ocultado.
- **Productos:** Banana es el producto más pedido (66 050 veces, tasa de recompra 0,844), y `produce` es el departamento con más artículos (1 328 302).
- **Fidelidad:** el producto más pedido no siempre es el más fiel. Entre los 739 productos con al menos 1 000 pedidos, el líder de recompra es Whole Organic Omega 3 Milk (0,863). Por pasillos, la leche tiene la tasa más alta (0,781) y las especias la más baja (0,152).
- **Clientes:** en promedio, cada cliente con detalle de productos compró 30,38 artículos.
- **Estructuración:** `concat`, `pivot_table` y `melt` permitieron cambiar la forma de las tablas, y los resultados quedaron guardados en `data/processed/` en Parquet (conserva los tipos) y CSV (se abre en Excel).